# Walk-Forward Validation

This notebook demonstrates expanding-window, horizon-aware walk-forward evaluation. It reports predictive metrics only; no trading backtest, positions, transaction costs, or profitability claims are included.

## Temporal Policy

Each fold fits preprocessing and models using its own training partition. Validation is used for model selection or LSTM early stopping. Test observations are evaluated only after fitting. A forecast horizon purges partition tails whose labels would reach into a later partition, and configured gaps provide additional embargo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.supervised import build_supervised_dataset
from ml.validation.aggregate import aggregate_fold_results
from ml.validation.baselines import evaluate_baselines_walk_forward
from ml.validation.config import WalkForwardConfig
from ml.validation.folds import generate_expanding_folds
from ml.validation.lstm import evaluate_lstm_walk_forward
from ml.validation.predictions import collect_predictions

## Load Data and Prepare Features

In [ ]:
raw_path = project_root / 'data' / 'raw' / 'AAPL.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    ohlcv = MarketDataIngestionService(YahooFinanceProvider()).ingest('AAPL', '2020-01-01', '2026-01-01')
# Use the existing supervised pipeline's feature matrix and target definitions.
supervised = build_supervised_dataset(ohlcv, target_type='regression', horizon=1)
X = pd.concat([supervised.X_train, supervised.X_validation, supervised.X_test], ignore_index=True)
y = pd.concat([supervised.y_train, supervised.y_validation, supervised.y_test], ignore_index=True)
dates = pd.concat([supervised.dates_train, supervised.dates_validation, supervised.dates_test], ignore_index=True)

## Configure and Inspect Folds

In [ ]:
config = WalkForwardConfig(initial_train_size=500, validation_size=100, test_size=100, step_size=100, forecast_horizon=1)
folds = generate_expanding_folds(len(X), config, dates)
for fold in folds:
    print(fold.fold, fold.train_dates[0], fold.train_dates[-1], fold.test_dates[0], fold.test_dates[-1])

## Baseline Walk-Forward Evaluation

In [ ]:
baseline_results = evaluate_baselines_walk_forward(X, y, dates, folds, task='regression', config=config)
baseline_summary = aggregate_fold_results(baseline_results)
baseline_summary

## LSTM Walk-Forward Evaluation

For a practical demonstration, use a modest lookback and training configuration. Each fold initializes a fresh LSTM and constructs train, validation, and test sequences independently.

In [ ]:
lstm_results = evaluate_lstm_walk_forward(X, y, dates, folds[:2], task='regression', config=config, lookback=20)
lstm_summary = aggregate_fold_results(lstm_results)
lstm_summary

## Out-of-Sample Predictions and Limitations

In [ ]:
predictions = collect_predictions(baseline_results + lstm_results)
predictions.head()

Metric variability across folds is important because market regimes change. A model that performs well in one period may not remain stable. These results are predictive evaluations only. No trading backtest or economic performance analysis has been performed yet.